In [1]:
# ----------------------------------------
# Project 05 - Microsoft Fabric Analytics Platform
# Notebook 08 - Gold Dimensions
# ----------------------------------------

from pyspark.sql import functions as F
from pyspark.sql.window import Window

silver_df = spark.table("silver.meter_readings")

print("Gold dimension build initialised.")

StatementMeta(, 52b3fd72-32d0-4f14-ac14-d79658e7c2bf, 3, Finished, Available, Finished, False)

Gold dimension build initialised.


In [2]:
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

print("Gold schema ready.")

StatementMeta(, 52b3fd72-32d0-4f14-ac14-d79658e7c2bf, 5, Finished, Available, Finished, False)

Gold schema ready.


In [3]:
dim_tariff_df = (
    silver_df
    .select("TariffType")
    .distinct()
    .withColumn(
        "TariffKey",
        F.when(F.col("TariffType") == "Std", F.lit(1))
        .when(F.col("TariffType") == "ToU", F.lit(2))
    )
    .select(
        "TariffKey",
        "TariffType"
    )
    .orderBy("TariffKey")
)

display(dim_tariff_df)

StatementMeta(, 52b3fd72-32d0-4f14-ac14-d79658e7c2bf, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 52b101a2-82e5-4dac-bd78-83560f235174)

In [4]:
(
    dim_tariff_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold.dim_tariff")
)

print("gold.dim_tariff written successfully.")

StatementMeta(, 52b3fd72-32d0-4f14-ac14-d79658e7c2bf, 7, Finished, Available, Finished, False)

gold.dim_tariff written successfully.


In [5]:
household_base_df = (
    silver_df
    .select(
        "HouseholdID",
        "tariffType"
    )
    .distinct()
)

StatementMeta(, 52b3fd72-32d0-4f14-ac14-d79658e7c2bf, 8, Finished, Available, Finished, False)

In [6]:
household_window = (
    Window
    .orderBy("HouseholdID")
)

dim_household_df = (
    household_base_df
    .withColumn(
        "HouseholdKey",
        F.row_number().over(household_window)
    )
    .withColumn(
        "TariffKey",
        F.when(F.col("TariffType") == "Std", F.lit(1))
        .when(F.col("TariffType") == "ToU", F.lit(2))
    )
    .select(
        "HouseholdKey",
        "HouseholdID",
        "TariffKey",
        "TariffType"
    )
)

StatementMeta(, 52b3fd72-32d0-4f14-ac14-d79658e7c2bf, 9, Finished, Available, Finished, False)

In [7]:
household_count = dim_household_df.count()

print("DIM HOUSEHOLD")
print("-" * 50)
print(f"Rows: {household_count:,}")

StatementMeta(, 52b3fd72-32d0-4f14-ac14-d79658e7c2bf, 10, Finished, Available, Finished, False)

DIM HOUSEHOLD
--------------------------------------------------
Rows: 5,561


In [8]:
duplicate_households = (
    dim_household_df
    .groupBy("HouseholdID")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Duplicate HouseholdIDs: {duplicate_households}")

StatementMeta(, 52b3fd72-32d0-4f14-ac14-d79658e7c2bf, 11, Finished, Available, Finished, False)

Duplicate HouseholdIDs: 0


In [9]:
(
    dim_household_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold.dim_household")
)

print("gold.dim_household written successfully.")

StatementMeta(, 52b3fd72-32d0-4f14-ac14-d79658e7c2bf, 12, Finished, Available, Finished, False)

gold.dim_household written successfully.


In [10]:
date_bounds = (
    silver_df
    .agg(
        F.min("ReadingDate").alias("MinDate"),
        F.max("ReadingDate").alias("MaxDate")
    )
    .collect()[0]
)

min_date = date_bounds["MinDate"]
max_date = date_bounds["MaxDate"]

print(f"Date range: {min_date} to {max_date}")

StatementMeta(, 52b3fd72-32d0-4f14-ac14-d79658e7c2bf, 13, Finished, Available, Finished, False)

Date range: 2011-11-23 to 2014-02-28


In [11]:
dim_date_base_df = (
    spark.sql(
        f"""
        SELECT explode(
            sequence(
                to_date('{min_date}'),
                to_date('{max_date}'),
                interval 1 day
            )
        ) AS Date
        """
    )
)

StatementMeta(, 52b3fd72-32d0-4f14-ac14-d79658e7c2bf, 14, Finished, Available, Finished, False)

In [15]:
dim_date_df = (
    dim_date_base_df
    .withColumn(
        "DateKey",
        F.date_format("Date", "yyyyMMdd").cast("int")
    )
    .withColumn(
        "Year",
        F.year("Date")
    )
    .withColumn(
        "Quarter",
        F.quarter("Date")
    )
    .withColumn(
        "MonthNumber",
        F.month("Date")
    )
    .withColumn(
        "MonthName",
        F.date_format("Date", "MMMM")
    )
    .withColumn(
        "YearMonth",
        F.date_format("Date", "yyyy-MM")
    )
    .withColumn(
        "DayOfMonth",
        F.dayofmonth("Date")
    )
    .withColumn(
        "DayOfWeekNumber",
        F.when(F.dayofweek("Date") == 1, 7)
         .otherwise(F.dayofweek("Date") - 1)
         .cast("int")
    )
    .withColumn(
        "DayOfWeekName",
        F.date_format("Date", "EEEE")
    )
    .withColumn(
        "IsWeekend",
        F.when(
            F.dayofweek("Date").isin(1, 7),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    .select(
        "DateKey",
        "Date",
        "Year",
        "Quarter",
        "MonthNumber",
        "MonthName",
        "YearMonth",
        "DayOfMonth",
        "DayOfWeekNumber",
        "DayOfWeekName",
        "IsWeekend"
    )
)

StatementMeta(, 52b3fd72-32d0-4f14-ac14-d79658e7c2bf, 18, Finished, Available, Finished, False)

In [16]:
date_count = dim_date_df.count()

print("DIM DATE")
print("-" * 50)
print(f"Rows: {date_count:,}")

display(dim_date_df.limit(10))

StatementMeta(, 52b3fd72-32d0-4f14-ac14-d79658e7c2bf, 19, Finished, Available, Finished, False)

DIM DATE
--------------------------------------------------
Rows: 829


SynapseWidget(Synapse.DataFrame, 1926ad8b-0c68-47be-8de0-e1f3d1d038da)

In [17]:
(
    dim_date_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold.dim_date")
)

print("gold.dim_date written successfully.")

StatementMeta(, 52b3fd72-32d0-4f14-ac14-d79658e7c2bf, 20, Finished, Available, Finished, False)

gold.dim_date written successfully.


In [18]:
time_slots_df = spark.range(0, 48)

StatementMeta(, 52b3fd72-32d0-4f14-ac14-d79658e7c2bf, 21, Finished, Available, Finished, False)

In [19]:
dim_time_df = (
    time_slots_df
    .withColumn(
        "Hour",
        F.floor(F.col("id") / 2).cast("int")
    )
    .withColumn(
        "Minute",
        F.when(
            (F.col("id") % 2) == 0,
            F.lit(0)
        ).otherwise(F.lit(30))
    )
    .withColumn(
        "TimeKey",
        (
            F.col("Hour") * 100 +
            F.col("Minute")
        ).cast("int")
    )
    .withColumn(
        "HalfHourSlot",
        (F.col("id") + 1).cast("int")
    )
    .withColumn(
        "TimeLabel",
        F.format_string(
            "%02d:%02d",
            F.col("Hour"),
            F.col("Minute")
        )
    )
    .withColumn(
        "TimeBand",
        F.when(F.col("Hour") < 6, "Overnight")
        .when(F.col("Hour") < 12, "Morning")
        .when(F.col("Hour") < 17, "Afternoon")
        .when(F.col("Hour") < 22, "Evening")
        .otherwise("Overnight")
    )
    .select(
        "TimeKey",
        "TimeLabel",
        "Hour",
        "Minute",
        "HalfHourSlot",
        "TimeBand"
    )
)

StatementMeta(, 52b3fd72-32d0-4f14-ac14-d79658e7c2bf, 22, Finished, Available, Finished, False)

In [20]:
print("DIM TIME")
print("-" * 50)
print(f"Rows: {dim_time_df.count()}")

display(dim_time_df)

StatementMeta(, 52b3fd72-32d0-4f14-ac14-d79658e7c2bf, 23, Finished, Available, Finished, False)

DIM TIME
--------------------------------------------------
Rows: 48


SynapseWidget(Synapse.DataFrame, 89f45997-166c-46ff-a6c2-421286c6ffcf)

In [21]:
(
    dim_time_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold.dim_time")
)

print("gold.dim_time written successfully.")

StatementMeta(, 52b3fd72-32d0-4f14-ac14-d79658e7c2bf, 24, Finished, Available, Finished, False)

gold.dim_time written successfully.


In [22]:
print("GOLD DIMENSION VALIDATION")
print("-" * 50)

print(
    f"dim_tariff: "
    f"{spark.table('gold.dim_tariff').count():,}"
)

print(
    f"dim_household: "
    f"{spark.table('gold.dim_household').count():,}"
)

print(
    f"dim_date: "
    f"{spark.table('gold.dim_date').count():,}"
)

print(
    f"dim_time: "
    f"{spark.table('gold.dim_time').count():,}"
)

StatementMeta(, 52b3fd72-32d0-4f14-ac14-d79658e7c2bf, 25, Finished, Available, Finished, False)

GOLD DIMENSION VALIDATION
--------------------------------------------------
dim_tariff: 2
dim_household: 5,561
dim_date: 829
dim_time: 48
